# `MT.PHY3` Maschinendynamik: **Kinematik eines Stanzautomaten**

## Einleitung

In den ersten Wochen haben Sie gelernt, die Kinematik (und Kinetik) ebener Systeme zu beschreiben.
Die bisherigen Aufgaben im Unterricht und Praktikum waren dabei so gestellt, dass sie rein analytisch lösbar waren, d. h., Sie konnten mit Papier und Stift zur Lösung gelangen.
Probleme in der Praxis sind jedoch meist so komplex, dass analytische Ansätze entweder nicht anwendbar oder nicht zielführend sind.
Zur Lösung praktischer Problemstellungen werden daher sehr oft numerische Methoden eingesetzt.

In diesem Praktikum erlernen Sie mithilfe des Computers durch Einsatz numerischer Methoden und Programmierung die Kinematik einer realen Maschine zu simulieren.
Konkret analysieren Sie die ebene Kinematik eines [Stanzautomaten](https://www.bruderer.com/de/produkte), wie er zum Stanzen und Umformen von Blechen eingesetzt wird.
Die Firma [Bruderer](https://www.bruderer.com/de) aus Frasnacht (TG) ist ein weltweit führender Hersteller von Stanzautomaten mit Nennkräften im Bereich $200 - 2500\,\mathrm{kN}$ und Hubzahlen von $100 - 2300\,\mathrm{min^{-1}}$.
Der in den Stanzautomaten von Bruderer eingesetzte Mechanismus ist im [Patent ](https://worldwide.espacenet.com/patent/search?q=EP0395964A1) auf den Seiten 2–3 sowie mit den Figuren 1 und 2 beschrieben.

<span style="color:Orange">Bitte bearbeiten Sie dieses Notebook zu zweit oder zu dritt.
Arbeiten Sie sich von oben nach unten vor und ergänzen Sie die Stellen, die mit `<edit>` markiert sind (Anweisungen sind wie hier orange eingefärbt).</span>
Das Notebook ist so aufgebaut, dass zu Beginn sehr vieles vorgeben ist.
Im weiteren Verlauf steigt die Schwierigkeit an, da Sie mehr und mehr selbst programmieren müssen.

Das Ziel dieses Praktikums liegt in der praktischen Anwendung der numerischen Methoden, die im [Python](https://www.python.org/doc/) Ökosystem mit verschiedenen Paketen (z.B., [NumPy](https://numpy.org/doc/stable/), [SciPy](https://docs.scipy.org/doc/scipy/), [Matplotlib](https://matplotlib.org/stable/index.html), ...) schlüsselfertig zur Verfügung stehen.
Die Funktionsweise und Grenzen dieser Algorithmen werden Sie im 4. Semester in der Vorlesung [Numerik](https://storagemkbdata.blob.core.windows.net/soe/moddescversions/Modulbeschreibung_t.BA.XXM5.NUM.22HS.pdf) erlernen.

### Lernziele

Sie können
 * die Kinematik ebener Mechanismen mittels numerischer Methoden berechnen.
 * den berechneten Bewegungsablauf visualisieren und verifizieren.
 * von den berechneten Grössen auf Lageebene Geschwindigkeiten und Beschleunigungen numerisch ableiten.
 

## Module

In [ ]:
import numpy as np                                          # numerical computing
from scipy.optimize import root_scalar, root                # finding roots of scalar and vector functions
import matplotlib.pyplot as plt                             # ploting
import matplotlib.animation as animation                    # animation
from IPython.display import HTML                            # movies within the notebook

In [ ]:
%matplotlib inline

## Inhalt

 0. [Kinematik Kolbenmaschine (Einführungsbeispiel)](#Kolbenmaschine)
 1. [Kinematik Stössel der Stanzautomat](#Stanzautomat)
 2. [Kinematik Ausgleichsmasse Stanzautomat (Challange)](#Ausgleichsmasse)

<a id="Kolbenmaschine"></a>
## 0. Kinematik Kolbenmaschine (Einführungsbeispiel)

Bevor Sie sich mit der komplexen Kinematik des Stanzautomaten beschäftigen, analysieren Sie als Einstieg die Kinematik eines einfachen Kurbeltriebs.

### 0.1 System

Die Kurbel $OP$ ist im Festlager $O$ drehend gelagert und treibt über den Pleuel $PQ$ den Kolben an, der im Loslager $Q$ vartikal geführt wird.
Die Auslenkung von Kurbel und Pleuel wird über die Winkel $\varphi$ und $\psi$ beschrieben, wobei der Kurbelwinkelverlauf $\varphi(t)$ als gegeben vorausgesetzt wird.
Für dieses Mechanismus können Sie alle numerisch berechneten Grössen analytisch nachrechnen und so verifizieren.
Der Kurbelradius $R$ und die Pleuellänge $L$ sind in der Skizze eingetragen und haben für die Berechnung folgende Werte:

| Variable | Mass                 |
|:-------:|:---------------------:|
|   $R$   |   $100\,\mathrm{mm}$  |
|   $L$   |   $400\,\mathrm{mm}$  |

<img src="kinematik_kurbeltrieb.svg" width="50%">

### 0.2 Parameter

<span style="color:Orange">Definieren Sie die geometrischen Parameter.</span>

> **Tipp**_: Definieren Sie dimensionsbehaftete Variablen in Programmen ausschliesslich in Standard SI-Einheiten!

In [ ]:
R = <edit>                                                  # radius of crank [m]
L = <edit>                                                  # length of connecting rod [m]

### 0.3 Analyse

Wir analysieren die Kinematik des Kurbeltriebs ausgehend vom Kurbelwinkel $\varphi$, den wir als vorgegeben betrachten.
Dazu diskretisieren wir den vollen Winkel $2\pi$ mit einer vorgegebenen Anzahl von Stützstellen $N$ und speichern diesen als Vektor `_ϕ` ab.

> _**Hinweis:**_ Diskretisierte Variablen kennzeichenen wir im Folgenden mit einem vorgesetzten Unterstrich `_`.

In [ ]:
N = 180                                                     # number of increments [-]
_ϕ = np.linspace(0, 2*np.pi, N, endpoint=False)             # discretized crank angle [rad]

Zur Analyse der Kinematik bilden wir uns eine Vektorkette von Orstvektoren, ausgehend vom Punkt $O$ über $P$ hin zu $Q$.

<span style="color:Orange">Definieren Sie hierzu die Funktion `r_OP` zum Berechnen des Ortsvektors $\mathbf{r}_{OP}$ als Funktion des Kurbelwinkels $\varphi$.</span>

> _**Hinweise**_:
>  * Vektoren (und Matrizen) werden in Python mit dem Numpy-Objekt [`array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html) erzeugt.
>  * Wir transponieren hier mit `.T`, damit das [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html#) in NumPy besser klappt (derzeit nicht so wichtig).

In [ ]:
def r_OP(ϕ):
    """Position vector from O to P"""
    return np.array([
        <edit>    ,
        <edit>    ]).T*R

<span style="color:Orange">Definieren Sie die Funktion `r_PQ` zum Berechnen des Ortsvektors $\mathbf{r}_{PQ}$ als Funktion des Pleuelwinkels $\psi$.</span>

In [ ]:
def r_PQ(ψ):
    """Position vector from P to Q"""
    return np.array([
        <edit>     ,
        <edit>     ]).T*L

<span style="color:Orange">Definieren Sie die Funktion `r_OQ` zum Berechnen des Ortsvektors $\mathbf{r}_{OQ}$ als Funktion der Winkel $\varphi$ und $\psi$.</span>

In [ ]:
def r_OQ(ϕ, ψ):
    """Position vector from O to Q"""
    return <edit>

Wegen dem Loslager in $Q$ ist der Ortsvektor $\mathbf{r}_{OQ}$ eingeschränkt.
Für einen vorgegebenen Kurbelwinkel $\varphi$ muss also der passende Pleuelwinkel $\psi$ so berechnet werden, dass die Lagerbingung in $Q$ erfüllt ist.

<span style="color:Orange">Formulieren Sie die Lagerbedingung an $\mathbf{r}_{OQ}$ als implizite Gleichung.
Ergänzen Sie die Funktion `ψ(ϕ)`, damit diese die implizite Gleichung für ein vorgebenes `ϕ` löst.</span>

> _**Hinweise**_:
>  * Wir lösen die implizite Gleichung numerisch mit dem Nullstellensucher [`root_scalar`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html) von [SciPy](https://docs.scipy.org/doc/scipy/).
>  * Wie in der Dokumentation zu [`root_scalar`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html) beschrieben, nutzen wir [`lambda`](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions) um aus der zweiparametrigen Funktion `r_OQ` eine Funktion zu erzeugen, die nur vom Parameter `ψ` abhängt.

In [ ]:
def ψ(ϕ, ψ0=0):
    """Rotation angle of connecting rod"""
    return root_scalar(lambda ψ : <edit>, x0=ψ0).root

Nun können wir die implizite Gleichung für $\psi$ numerisch für jeden diskreten Winkel $\varphi$ auswerten.

> _**Hinweis**_: Wir verwenden in Python die [List Comprehension](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions) um in den eckigen Klammern auf kompakte Art direkt eine Liste zu erzeugen.

In [ ]:
_ψ = np.array([ψ(ϕ) for ϕ in _ϕ])

In einem letzten Schritt werden die Ortsvektoren für die $N$ diskreten Stützstellen ausgewertet und als $(N \times 2)$-Matrizen abgespeichert.

In [ ]:
_r_OP = r_OP(_ϕ)
_r_OQ = r_OQ(_ϕ, _ψ)

### 0.4 Plot und Anmimation

Das kinematische Problem für den Kurbeltrieb ist nun gelöst.
Zum Verständnis, zur Verifikation (und zur Problemlösung) ist es extrem hilfreich sich die Lösung zu visualisieren.
Wir plotten zuerst die Trajektorien der Punkte $P$ und $Q$.

In [ ]:
fig0, ax0 = plt.subplots(1, 1, num='Kurbeltrieb', clear=True)
ax0.axis('equal');

ax0.plot(_r_OP[:, 0], _r_OP[:, 1], alpha=.5, label='Trajektorie $P$')
ax0.plot(_r_OQ[:, 0], _r_OQ[:, 1], alpha=.5, label='Trajektorie $Q$')

ax0.legend();

Nun zeichnen wir für den Ausganszustand $\varphi=0$ die Kurbel und den Pleuel ein.
Beide Körper stellen wir vereinfacht als Linie dar, die die Punkten $O$, $P$ und $Q$ verbinden.

In [ ]:
crank, = ax0.plot([0, _r_OP[0, 0]], [0, _r_OP[0, 1]], 'k.-')
rod, = ax0.plot([_r_OP[0, 0], _r_OQ[0, 0]], [_r_OP[0, 1], _r_OQ[0, 1]], 'k.-')
ax0.figure

In [Matplotlib](https://matplotlib.org/stable/index.html) kann der Mechanismus mit der Klasse [`FuncAnimation`](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html) animiert werden.
Die Animation wird generiert, indem Frame für Frame die Funktion `update0` aufgerufen wird.
In `update0` werden die Koordinaten der Endpunkte beider Linien mit der Methode [`set_data`](https://matplotlib.org/stable/api/_as_gen/matplotlib.lines.Line2D.html#matplotlib.lines.Line2D.set_data) aktualisiert.
Die Klasse [`HTML`](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.HTML) bettet die Animation im Jupyter Notebook ein.

> _**Beachte**_: Die Funktion `update0` muss alle aktualisierten Objekte (Artists) als Rückgabeparameter ausgeben.

In [ ]:
def update0(i):
    """Update function for animation"""
    
    # Update coordinates
    crank.set_data([0, _r_OP[i, 0]], [0, _r_OP[i, 1]])
    rod.set_data([_r_OP[i, 0], _r_OQ[i, 0]], [_r_OP[i, 1], _r_OQ[i, 1]])

    return crank, rod

ani0 = animation.FuncAnimation(fig0, update0, N, interval=20, blit=True)

HTML(ani0.to_jshtml())

Sie haben gelernt die Kinematik des Kurbeltriebs numerisch zu berechnen und visualisieren.
Nun übertragen Sie das am einfachen Mechanismus Gelernte auf ein komplexeres Problem, nämlich der Kinematik des Stanzautomaten.
Das Vorgehen bleibt prinzipiell gleich.

<a id="Stanzautomat"></a>
## 1. Kinematik Stanzautomat

### 1.1 System

Für die Analyse der Kinematik des Stanzautomaten wird das abgebildete Halbmodell verwendet.
Die Kurbel $OA$ ist im Festlager $O$ drehend gelagert und treibt über den Pleuel $AB$ den Hebel $BC$ an, der im Loslager $B$ vertikal und im Loslager $C$ horizontal gelagert ist.
Der Stössel ist in den Loslagern $G$ und $F$ vertikal geführt und üer die Drucksäulse $DE$ mit dem Hebel $BC$ verbunden.
Die Auslenkung von Kurbel, Pleuel, Hebel und Drucksäule wird über die Winkel $\alpha$, ... $\delta$ beschrieben, wobei der Kurbelwinkielverlauf $\alpha(t)$ als gegeben vorausgesetzt wird.
Die relevanten Masse $r$, $a$, ..., $f$ sind in der Skizze eingetragen und definiert als

| Variable | Mass                 |
|:-------:|:---------------------:|
|   $r$   |    $50\,\mathrm{mm}$  |
|   $a$   |   $400\,\mathrm{mm}$  |
|   $b$   |   $150\,\mathrm{mm}$  |
|   $c$   |   $500\,\mathrm{mm}$  |
|   $d$   |   $250\,\mathrm{mm}$  |
|   $e$   |   $100\,\mathrm{mm}$  |
|   $f$   |   $500\,\mathrm{mm}$  |

<img src="kinematik_stanzautomat.svg" width="100%">

### 1.2 Parameter

<span style="color:Orange">Definieren Sie die geometrischen Parameter.</span>

In [ ]:
r = <edit>                                                  # distance OA [m]
a = <edit>                                                  # distance AB [m]
b = <edit>                                                  # distance BD [m]
c = <edit>                                                  # distance BC [m]
d = <edit>                                                  # distance DE [m]
e = <edit>                                                  # distance FE in x-direction [m]
f = <edit>                                                  # distance OE in y-direction [m]

### 1.3. Analyse

<span style="color:Orange">Diskretisieren Sie den treibenden Kurbelwinkels $\alpha$ mit $N = 180$ äquidistaten Stützstellen</span>

In [ ]:
<edit>                                                      # number of increments
<edit>                                                      # vector of angle increments α

<span style="color:Orange">Definieren Sie die Funktion `r_OA` zum Berechnen des Ortsvektors $\mathbf{r}_{OA}$ als Funktion des Kurbelwinkels $\alpha$.</span>

In [ ]:
def r_OA(α):
    """Position vector from O to A"""
    return <edit>

In [ ]:
_r_OA = r_OA(_α)

<span style="color:Orange">Definieren Sie die Funktion `r_AB` zum Berechnen des Ortsvektors $\mathbf{r}_{AB}$ als Funktion des Pleuelwinkels $\beta$.</span>

In [ ]:
def r_AB(β):
    """Position vector from A to B"""
    return <edit>

<span style="color:Orange">Definieren Sie die Funktion `r_OB` zum Berechnen des Ortsvektors $\mathbf{r}_{OB}$ als Funktion der Winkel $\alpha$ und $\beta$.</span>

In [ ]:
def r_OB(α, β):
    """Position vector from O to B"""
    return <edit>

<span style="color:Orange">Formulieren Sie die Lagerbedingung an $\mathbf{r}_{OB}$ als implizite Gleichung.
Ergänzen Sie die Funktion `β(α)`, damit diese die implizite Gleichung für ein vorgebenes `α` löst.</span>

In [ ]:
def β(α, β0=0):
    """Rotation angle of connecting rod"""
    return root_scalar(lambda β : <edit>, x0=β0).root

In [ ]:
_β = np.array([β(α) for α in _α])
_r_OB = r_OB(_α, _β)

<span style="color:Orange">Definieren Sie analog die Funktionen `r_BC` und `r_OC` zum Berechnen des Ortsvektors $\mathbf{r}_{OC}$ als Funktion der Winkel $\alpha$, $\beta$ und $\gamma$.</span>

In [ ]:
def r_BC(γ):
    """Position vector from B to C"""
    return <edit>

def r_OC(α, β, γ):
    """Position vector from O to C"""
    return <edit>

<span style="color:Orange">Formulieren Sie die Lagerbedingung an $\mathbf{r}_{OC}$ als implizite Gleichung.
Ergänzen Sie die Funktion `γ(α, β)`, damit diese die implizite Gleichung für ein vorgebene `α` und `β` löst.</span>

In [ ]:
def γ(α, β, γ0=0):
    """Rotation angle of lever"""
    return <edit>

In [ ]:
_γ = np.array([γ(α, β) for α, β in zip(_α, _β)])
_r_OC = r_OC(_α, _β, _γ)

<span style="color:Orange">Definieren Sie analog die Funktionen `r_BD` und `r_OD` zum Berechnen des Ortsvektors $\mathbf{r}_{OD}$ als Funktion der Winkel $\alpha$, $\beta$ und $\gamma$.</span>

In [ ]:
def r_BD(γ):
    """Position vector from B to D"""
    return <edit>

def r_OD(α, β, γ):
    """Position vector from O to D"""
    return <edit>

In [ ]:
_r_OD = r_OD(_α, _β, _γ)

<span style="color:Orange">Definieren Sie analog die Funktionen `r_DE` und `r_OE` zum Berechnen des Ortsvektors $\mathbf{r}_{OE}$ als Funktion der Winkel $\alpha$, $\beta$, $\gamma$ und $\delta$.</span>

In [ ]:
def r_DE(δ):
    """Position vector from D to E"""
    return <edit>

def r_OE(α, β, γ, δ):
    """Position vector from O to E"""
    return <edit>

<span style="color:Orange">Formulieren Sie die Lagerbedingung an $\mathbf{r}_{OE}$ als implizite Gleichung.
Ergänzen Sie die Funktion `δ(α, β, γ)`, damit diese die implizite Gleichung für ein vorgebene `α`, `β`  und `γ` löst.</span>

In [ ]:
def δ(α, β, γ, δ0=0):
    """Rotation angle of pressure column"""
    return <edit>

In [ ]:
_δ = np.array([δ(α, β, γ) for α, β, γ in zip(_α, _β, _γ)])
_r_OE = r_OE(_α, _β, _γ, _δ)

### 1.4 Plot und Animation

<span style="color:Orange">Zeichnen Sie die Trajektorien der Punkte $A$, ..., $E$.</span>

In [ ]:
fig1, ax1 = plt.subplots(1, 1, num='Stamping Machine', clear=True)
ax1.axis('equal');

ax1.plot(<edit>     , <edit>     , alpha=.5, label='Trajektorie $A$')
ax1.plot(<edit>     , <edit>     , alpha=.5, label='Trajektorie $B$')
ax1.plot(<edit>     , <edit>     , alpha=.5, label='Trajektorie $C$')
ax1.plot(<edit>     , <edit>     , alpha=.5, label='Trajektorie $D$')
ax1.plot(<edit>     , <edit>     , alpha=.5, label='Trajektorie $E$')

ax1.legend();

<span style="color:Orange">Zeichnen Sie den Kurbeltrieb $OA$, den Pleuel $AB$, den Hebel $BC$ und die Drucksäule $DE$ für den Kurbelwinkel $\alpha=0$ vereinfacht als Linien.</span>

In [ ]:
crank, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-')
rod, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-')
lever, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-')
column, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-')

ax1.figure

<span style="color:Orange">Animieren Sie den Kurbeltrieb, indem Sie die Funktion `update1` ergänzen.</span>

In [ ]:
def update1(i):
    """Update function for animation"""
    
    # Update coordinates
    crank.set_data([<edit>, <edit>], [<edit>, <edit>])
    rod.set_data([<edit>, <edit>], [<edit>, <edit>])
    lever.set_data([<edit>, <edit>], [<edit>, <edit>])
    column.set_data([<edit>, <edit>], [<edit>, <edit>])
    
    return <edit>, <edit>, <edit>, <edit>

ani1 = animation.FuncAnimation(fig1, update1, N, interval=20, blit=True)

HTML(ani1.to_jshtml())

<a id="Ausgleichsmasse"></a>
## 2. Kinematik Stanzautomat mit Ausgleichsmasse (Challenge)

### 2.1 System

Der Stössel des Stanzautomaten vollführt beim Stanzen eine extrem schnelle Auf- und Abwärtsbewegung.
Um eine gute Laufruhe zu erzielen, muss diese vertikale Bewegung des Stössels und auch die horizontale Bewegung des Pleuels mit Ausgleichsmassen kompensiert werden.
Bruderer hat zum Massenausgleich einen raffinierten Mechanimus entwickelt, dessen Kinematik Sie in dieser Zusatzaufgabe analysieren (siehe Skizze).
Die Ausgleichsmasse ist bei $L$ am Kippehebel aufgehängt, der in $K$ gelagert ist und über den oberen Pleuel bei $J$ von der Kurbel angetrieben wird.
Bei $M$ ist die Ausgleichsmasse über einen Stab mit dem verlängerten Ende $H$ des unteren Pleuels verbunden.
Die Verdrehung von Pleuel, Kipphebel, Ausgleichsmasse und Stab sind mit den Winkeln $\epsilon$, $\theta$, $\varphi$ und $\psi$ beschrieben.

Prinzipiell ist das Vorgeben sehr ähnlich wie in der vorherigen Aufgabe 1, es gibt hier jedoch eine zusätzliche Schwierigkeit:
Die kinematische Kette $A \to J \to K \to L \to M \to H \to A$ ist geschlossen.
Daher kann man nicht mehr einfach in einer Richtung entlang der kinematischen Kette rechnen, sondern muss mehrere Pfade berücksichtigen.

Die relevanten Masse $g$,..., $l$ sind in der Skizze eingetragen und definiert als

| Variable | Mass                 |
|:-------:|:---------------------:|
|   $g$   |   $160\,\mathrm{mm}$  |
|   $h$   |   $550\,\mathrm{mm}$  |
|   $k$   |   $320\,\mathrm{mm}$  |
|   $l$   |   $960\,\mathrm{mm}$  |

<img src="kinematik_stanzautomat_ausgleichsmasse.svg" width="100%">

### 2.2 Parameter

<span style="color:Orange">Definieren Sie die geometrischen Parameter.</span>

In [ ]:
g = <edit>                                                  # distance BH [m]
h = <edit>                                                  # distance HM or distance JL [m]
k = <edit>                                                  # distance JK or distance OK in x-direction [m]
l = <edit>                                                  # distance ML [m]

<span style="color:Orange">Definieren Sie die Funktion `r_AJ` zum Berechnen des Ortsvektors $\mathbf{r}_{AJ}$ als Funktion des Winkels $\varepsilon$.</span>

In [ ]:
def r_AJ(ε):
    """Position vector from A to J"""
    <edit>

<span style="color:Orange">Definieren Sie die Funktion `r_OJ1` zum Berechnen des Ortsvektors $\mathbf{r}_{OJ}$ als Funktion der Winkel $\alpha$ und $\varepsilon$.</span>
Das beschreibt den ersten Pfad $O \to A \to J$.

In [ ]:
def r_OJ1(α, ε):
    """Position vector from O to J"""
    <edit>

<span style="color:Orange">Definieren Sie die Variable `r_OK` für den Ortsvektor $\mathbf{r}_{OK}$.</span>

In [ ]:
r_OK = <edit>

<span style="color:Orange">Definieren Sie die Funktion `r_KJ` zum Berechnen des Ortsvektors $\mathbf{r}_{KJ}$ als Funktion des Winkels $\theta$.</span>

In [ ]:
def r_KJ(θ):
    """Position vector from K to J"""
    <edit>

<span style="color:Orange">Definieren Sie die Funktion `r_OJ2` zum Berechnen des Ortsvektors $\mathbf{r}_{OJ}$ als Funktion des Winkels $\theta$.</span>
Das beschreibt den zweiten Pfad $O \to K \to J$.

In [ ]:
def r_OJ2(θ):
    """Position vector from O to J"""
    <edit>

<span style="color:Orange">Formulieren Sie eine Bedingung an $\mathbf{r}_{OJ}$ als implizite, vektorielle Gleichung.
Ergänzen Sie die Funktion `εθ(α)`, damit diese die implizite Gleichung für ein vorgebenes `α` löst.</span>

> _**Hinweis**_:
>  * Da die implizite Gleichung nun vektorwertig ist, benötigen wir zur numerisch Lösung nun den Nullstellensucher [`root`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root.html) von [SciPy](https://docs.scipy.org/doc/scipy/).

In [ ]:
def εθ(α, ε0=0, θ0=0):
    """Rotation angle of upper connecting rod and upper lever"""
    return root(lambda x : r_OJ1(α, x[0]) - r_OJ2(x[1]), x0=[ε0, θ0]).x

In [ ]:
_ε, _θ = np.array([εθ(α) for α in _α]).T

In [ ]:
_r_OJ = r_OJ2(_θ)

<span style="color:Orange">Definieren Sie analog wie oben die Funktionen `r_KL` und `r_OL` zum Berechnen des Ortsvektors $\mathbf{r}_{OL}$ als Funktion des Winkels $\theta$.</span>

In [ ]:
def r_KL(θ):
    """Position vector from K to L"""
    <edit>

def r_OL(θ):
    """Position vector from O to L"""
    <edit>

In [ ]:
_r_OL = r_OL(_θ)

<span style="color:Orange">Definieren Sie die Funktionen `r_AH` und `r_OH` zum Berechnen des Ortsvektors $\mathbf{r}_{OH}$ als Funktion de Winkel $\alpha$ und $\beta$.</span>

In [ ]:
def r_AH(β):
    """Position vector from A to H"""
    <edit>

def r_OH(α, β):
    """Position vector from O to J"""
    <edit>

In [ ]:
_r_OH = r_OH(_α, _β)

<span style="color:Orange">Definieren Sie die Funktionen `r_HM` und `r_OM1` zum Berechnen des Ortsvektors $\mathbf{r}_{OM}$ als Funktion der Winkel $\alpha$, $\beta$ und $\psi$.</span>
Das beschreibt den ersten Pfad $O \to A \to H \to M$.

In [ ]:
def r_HM(ψ):
    """Position vector from H to M"""
    <edit>

def r_OM1(α, β, ψ):
    """Position vector from O to M"""
    <edit>

<span style="color:Orange">Definieren Sie die Funktionen `r_LM` und `r_OM2` zum Berechnen des Ortsvektors $\mathbf{r}_{OM}$ als Funktion der Winkel $\theta$ und $\phi$.</span>
Das beschreibt den zweiten Pfad $O \to A \to J \to L \to M$.

In [ ]:
def r_LM(ϕ):
    """Position vector from L to M"""
    <edit>

def r_OM2(θ, ϕ):
    """Position vector from O to M"""
    <edit>

<span style="color:Orange">Formulieren Sie eine Bedingung an $\mathbf{r}_{OM}$ als implizite, vektorielle Gleichung.
Ergänzen Sie die Funktion `ϕψ(α, β, θ)`, damit diese die implizite Gleichung für vorgebene `α`, `β` und `θ` löst.</span>

In [ ]:
def ϕψ(α, β, θ, ϕ0=0, ψ0=0):
    """Rotation angle of upper connecting rod and upper lever"""
    <edit>

In [ ]:
_ϕ, _ψ = np.array([ϕψ(α, β, θ) for α, β, θ in zip(_α, _β, _θ)]).T

In [ ]:
_r_OM = r_OM2(_θ, _ϕ)

### 2.4 Plot und Animation

<span style="color:Orange">Ergänzen Sie die bestehende Achse `ax1` indem Sie die Trajektorien der Punkte $H$, ..., $M$ einzeichnen.</span>

In [ ]:
<edit>
<edit>
<edit>
<edit>

ax1.legend()
ax1.figure

<span style="color:Orange">Zeichnen Sie den oberen Pleuel $AJ$, den Kipphebel $JL$, den Stab $HM$ und die Ausgleichsmasse $LM$ für den Kurbelwinkel $\alpha=0$ vereinfacht als Linien.</span>

In [ ]:
mass, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-', alpha=.5, linewidth=10)
rod_upper, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-')
lever_upper, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-')
rod_lower, = ax1.plot([<edit>, <edit>], [<edit>, <edit>], 'k.-')

ax1.figure

<span style="color:Orange">Ergänzen Sie die Funktion `update2` um die Kinematik der Stanzmaschine mit Massenausgleich zu animieren.</span>

In [ ]:
def update2(i):
    """Update function for animation"""
    
    # Update coordinates
    crank.set_data([<edit>, <edit>], [<edit>, <edit>])
    rod.set_data([<edit>, <edit>], [<edit>, <edit>])
    lever.set_data([<edit>, <edit>], [<edit>, <edit>])
    column.set_data([<edit>, <edit>], [<edit>, <edit>])
    rod_upper.set_data([<edit>, <edit>], [<edit>, <edit>])
    lever_upper.set_data([<edit>, <edit>], [<edit>, <edit>])
    rod_lower.set_data([<edit>, <edit>], [<edit>, <edit>])
    mass.set_data([<edit>, <edit>], [<edit>, <edit>])
    
    return crank, rod, lever, column, rod_upper, lever_upper, rod_lower, mass

ani1 = animation.FuncAnimation(fig1, update2, N, interval=20, blit=True)

HTML(ani1.to_jshtml())